# 25882 AI-powered Investment and Risk Management — Assessment 1: Empirical Assignment

**Group members (name, student ID):**
- TODO: Name 1, Student ID 1
- TODO: Name 2, Student ID 2 *(delete this line if working alone)*


## Part B choices

TODO — once selected, state explicitly which two extensions we are attempting, e.g.:

> We attempt **B1 (Portfolio optimisation)** from Category B and **A2 (Factor analysis)** from Category A,
> satisfying the requirement that the two options come from different categories and at least one comes
> from Category A or B.


In [ ]:
import importlib.util
import subprocess
import sys


def _ensure_installed(packages):
    """
    Cheap safety net for `Kernel -> Restart & Run All` on a machine where the
    notebook's kernel does not match the environment `pip install -r
    requirements.txt` was run into (a common Jupyter/conda mix-up). Checks
    each package's presence in *this* kernel only -- not its version -- and
    installs only what is actually missing, into `sys.executable` so it
    cannot disagree with itself. A no-op, and silent, when everything is
    already available -- the expected case in a correctly set-up environment.
    """
    missing = [p for p in packages if importlib.util.find_spec(p) is None]
    if missing:
        print(f"Installing missing packages into this kernel: {missing}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])


_ensure_installed(["numpy", "pandas", "yfinance"])

import warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import yfinance as yf

warnings.filterwarnings("ignore", category=FutureWarning)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

DATA_DIR = Path("data_cache")
DATA_DIR.mkdir(exist_ok=True)

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
print(f"Notebook environment set up. Random seed fixed at {RANDOM_SEED}.")


## A1 — Universe selection and data acquisition

### Asset universe and justification

| Ticker | Name | Sector | Exchange | Currency |
|---|---|---|---|---|
| AAPL | Apple Inc. | Information Technology | NASDAQ | USD |
| JPM | JPMorgan Chase & Co. | Financials | NYSE | USD |
| XOM | ExxonMobil Corp. | Energy | NYSE | USD |
| JNJ | Johnson & Johnson | Health Care | NYSE | USD |
| 7203.T | Toyota Motor Corp. | Consumer Discretionary (Automobiles) | Tokyo Stock Exchange | JPY |

We chose four large-cap US stocks spanning four distinct GICS sectors (technology, financials, energy,
healthcare) plus one Japan-listed stock (Toyota, 7203.T) to satisfy two goals at once. First, four
distinct sectors avoid the degenerate, near-perfectly-correlated universe a single-sector selection
would produce, which the brief warns would flatten the diversification and optimisation results in
Part B. Second, Toyota is listed on the Tokyo Stock Exchange and trades in JPY, giving us a genuine
cross-currency, cross-trading-calendar asset — required by the brief, and necessary to make the
currency-alignment and holiday-calendar corrections in Section A2 non-vacuous. A different choice —
five US large-caps in USD, say — would have prevented us from ever observing an FX or calendar-alignment
effect at all, and would have understated how much diversification the equal-weight benchmark in A4
can actually deliver.


In [ ]:
# --- Universe and sample period ---
TICKERS = ["AAPL", "JPM", "XOM", "JNJ", "7203.T"]

# Fixed, hard-coded date range (not "today") so the notebook is exactly
# reproducible on re-run: the price data requested does not depend on when
# the notebook happens to be executed. This spans just over 9 years, safely
# above the 8-year minimum.
PRICE_START = "2016-01-01"
PRICE_END = "2025-01-01"

# Recorded once, manually, at the time of the first successful live download
# below -- this is metadata about *when we pulled the data*, not a parameter
# that should silently change what gets downloaded on a later re-run.
DOWNLOAD_DATE_RECORDED = "2026-09-15"  # TODO: update to the actual date you first run this successfully

print(f"Universe: {TICKERS}")
print(f"Requested date range: {PRICE_START} to {PRICE_END}")


In [ ]:
def load_price_panel(tickers, start, end, cache_dir=DATA_DIR, force_download=False):
    """
    Download daily unadjusted and dividend/split-adjusted close prices for
    `tickers` between `start` and `end`, and return them as two clean, wide,
    date-aligned DataFrames (one column per ticker).

    Used by every later section of this notebook -- built once here, reused
    throughout, per the assignment's instruction not to re-implement this.

    Behaviour
    ---------
    * yfinance now defaults to auto_adjust=True, which returns only an
      adjusted Close and discards the raw price. We call it with
      auto_adjust=False explicitly so both series are available, because
      Section A2 needs both.
    * A ticker that returns nothing, or returns an all-NaN Close column, is
      dropped and reported -- *before* any row-wise NaN handling. Dropping
      NaN rows first would let one broken ticker delete the entire sample.
    * On success, the two wide panels are cached to `cache_dir` as CSV files.
      On a later call (or a later notebook re-run) with the cache already
      present, the function reads the cache directly and does not touch the
      network at all.
    * If a live download raises (network error, rate limit, vendor outage --
      "which happens", per the brief) the function falls back to the local
      cache if one exists, so the notebook degrades gracefully rather than
      crashing on a clean re-run at marking time.

    Parameters
    ----------
    tickers : list of str
    start, end : str, "YYYY-MM-DD"
    cache_dir : Path
    force_download : bool
        Skip a fresh-cache check and always attempt a live download first
        (still falls back to the cache on failure).

    Returns
    -------
    raw_close : DataFrame, date-indexed, one column per surviving ticker
    adj_close : DataFrame, date-indexed, one column per surviving ticker
    log : dict
        Provenance metadata: source, timestamp, tickers requested vs
        returned, yfinance version.
    """
    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    raw_path = cache_dir / "raw_close.csv"
    adj_path = cache_dir / "adj_close.csv"

    if not force_download and raw_path.exists() and adj_path.exists():
        raw_close = pd.read_csv(raw_path, index_col=0, parse_dates=True)
        adj_close = pd.read_csv(adj_path, index_col=0, parse_dates=True)
        log = {
            "source": "local cache",
            "path": str(cache_dir),
            "tickers": list(raw_close.columns),
        }
        print(f"Loaded cached panel from {cache_dir}/ "
              f"(delete raw_close.csv / adj_close.csv there to force a fresh download).")
        return raw_close, adj_close, log

    try:
        data = yf.download(tickers, start=start, end=end, auto_adjust=False,
                            group_by="ticker", progress=False, threads=True)
        if data.empty:
            raise ValueError("yfinance returned an empty frame for every ticker")

        raw_cols, adj_cols = {}, {}
        for t in tickers:
            try:
                sub = data[t]
            except KeyError:
                print(f"  WARNING: no data at all returned for {t}; dropping from panel")
                continue
            if sub["Close"].dropna().empty:
                print(f"  WARNING: {t} returned an all-NaN Close column; dropping from panel")
                continue
            raw_cols[t] = sub["Close"]
            adj_cols[t] = sub["Adj Close"]

        if not raw_cols:
            raise ValueError("Every requested ticker came back empty or all-NaN")

        raw_close = pd.DataFrame(raw_cols).sort_index()
        adj_close = pd.DataFrame(adj_cols).sort_index()

        raw_close.to_csv(raw_path)
        adj_close.to_csv(adj_path)

        log = {
            "source": "yfinance (live download)",
            "download_timestamp": datetime.now().isoformat(timespec="seconds"),
            "yfinance_version": getattr(yf, "__version__", "unknown"),
            "requested_tickers": list(tickers),
            "returned_tickers": list(raw_close.columns),
            "requested_start": start,
            "requested_end": end,
        }
        print(f"Downloaded fresh data and cached it to {cache_dir}/.")
        return raw_close, adj_close, log

    except Exception as exc:
        print(f"Live download failed ({exc!r}).")
        if raw_path.exists() and adj_path.exists():
            print("Falling back to the previously cached local files so the "
                  "notebook can still run end to end.")
            raw_close = pd.read_csv(raw_path, index_col=0, parse_dates=True)
            adj_close = pd.read_csv(adj_path, index_col=0, parse_dates=True)
            log = {
                "source": "local cache (after a failed live download)",
                "path": str(cache_dir),
            }
            return raw_close, adj_close, log
        raise RuntimeError(
            f"No live data available and no local cache found at {cache_dir}/. "
            "Cannot proceed -- see the note on reproducibility in the assignment brief."
        ) from exc


In [ ]:
raw_close, adj_close, download_log = load_price_panel(TICKERS, PRICE_START, PRICE_END)

print("\nDownload log:")
for k, v in download_log.items():
    print(f"  {k}: {v}")


In [ ]:
def summarize_panel(price_df, label="", min_years=8):
    """
    Report, per asset: first date, last date, observation count, and years
    spanned -- and flag any asset whose history falls short of `min_years`.

    This is the "inspect what arrived before using it" step: it does not
    silently trust the download, it shows the evidence.
    """
    summary = pd.DataFrame({
        "first_date": price_df.apply(lambda s: s.dropna().index.min()),
        "last_date": price_df.apply(lambda s: s.dropna().index.max()),
        "n_obs": price_df.count(),
    })
    summary["years_span"] = (summary["last_date"] - summary["first_date"]).dt.days / 365.25

    print(f"--- {label} ---")
    display(summary)

    short = summary[summary["years_span"] < min_years]
    if not short.empty:
        print(f"WARNING: history shorter than {min_years} years for: {list(short.index)}")
    else:
        print(f"All assets meet the {min_years}-year minimum.")
    return summary


raw_summary = summarize_panel(raw_close, label="Raw close -- inspection", min_years=8)
adj_summary = summarize_panel(adj_close, label="Adjusted close -- inspection", min_years=8)


### A1 discussion

TODO once run with live data: comment on the actual date ranges and observation counts returned --
in particular, whether Toyota's calendar (Tokyo trading days) gives a different `n_obs` from the
US tickers even over the same nominal date range, and whether any ticker's history fell short of
the 8-year requirement.

**Note on this notebook's execution environment:** the sandbox used to draft this notebook has no
outbound network access to Yahoo Finance, so `load_price_panel()` above could not perform a live
download here -- its logic was verified separately against a mocked `yfinance.download` covering
(1) a normal multi-ticker download, (2) a ticker returning an all-NaN column being dropped before
any row-level NaN handling, (3) a cache-hit skipping the network entirely, and (4) a live-download
failure falling back to a previously cached panel. Run **Kernel → Restart & Run All** on a machine
with internet access to perform the actual download, populate `data_cache/raw_close.csv` and
`data_cache/adj_close.csv`, and replace this note with the real inspection results.
